In [1]:
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor

In [2]:
data = fetch_openml(
    name="house_prices",
    as_frame=True
)

In [3]:
X = data.data
y = data.target

In [4]:
print("First 5 rows:")
print(X.head())

print("\nShape:")
print(X.shape)

print("\nData types:")
print(X.dtypes)

print("\nMissing values:")
print(X.isnull().sum())

First 5 rows:
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... ScreenPorch PoolArea PoolQC Fence MiscFeature  \
0         Lvl    AllPub  ...           0        0    NaN   NaN         NaN   
1         Lvl    AllPub  ...           0        0    NaN   NaN         NaN   
2         Lvl    AllPub  ...           0        0    NaN   NaN         NaN   
3         Lvl    AllPub  ...           0        0    NaN   NaN         NaN   
4         Lvl    AllPub  ...           0        0    NaN   NaN         NaN   

  MiscVal MoSold  YrSold  SaleType  SaleCondition  
0       0      2    

In [5]:
# Separate numerical and categorical features
numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns

categorical_features = X.select_dtypes(
    include=["object"]
).columns

print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 37
Number of categorical features: 43


C:\Users\Hp\AppData\Local\Temp\ipykernel_2188\1697451014.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [6]:
# Fill missing numerical values with the median

numerical_transformer = SimpleImputer(strategy="median")

In [7]:
# Fill missing categories and convert them to numbers

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [8]:
# Apply the correct preprocessing to each feature type

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [9]:
# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 1168
Testing rows: 292


In [10]:
# Train the Linear Regression model

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)
print("Linear Regression training complete.")

Linear Regression training complete.


In [11]:
# Train the Random Forest model

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor())
    ]
)

random_forest_model.fit(X_train, y_train)
print("Random Forest training complete.")

Random Forest training complete.


In [12]:
# Train the XGBoost model

xgboost_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor())
    ]
)

xgboost_model.fit(X_train, y_train)
print("XGBoost training complete.")

XGBoost training complete.


In [13]:
# Make predictions on the test data

linear_predictions = linear_model.predict(X_test)
random_forest_predictions = random_forest_model.predict(X_test)
xgboost_predictions = xgboost_model.predict(X_test)

In [14]:
# Calculate RMSE for each model

linear_rmse = mean_squared_error(
    y_test,
    linear_predictions
) ** 0.5

random_forest_rmse = mean_squared_error(
    y_test,
    random_forest_predictions
) ** 0.5

xgboost_rmse = mean_squared_error(
    y_test,
    xgboost_predictions
) ** 0.5

print(f"Linear Regression RMSE: KES {linear_rmse:,.0f}")
print(f"Random Forest RMSE: KES {random_forest_rmse:,.0f}")
print(f"XGBoost RMSE: KES {xgboost_rmse:,.0f}")

Linear Regression RMSE: KES 31,270
Random Forest RMSE: KES 29,427
XGBoost RMSE: KES 28,339


In [15]:
# Calculate R² for each model

linear_r2 = r2_score(y_test, linear_predictions)
random_forest_r2 = r2_score(y_test, random_forest_predictions)
xgboost_r2 = r2_score(y_test, xgboost_predictions)

print(f"Linear Regression R²: {linear_r2:.3f}")
print(f"Random Forest R²: {random_forest_r2:.3f}")
print(f"XGBoost R²: {xgboost_r2:.3f}")

Linear Regression R²: 0.873
Random Forest R²: 0.887
XGBoost R²: 0.895


In [16]:
# Compare the three models

results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost"
    ],
    "RMSE": [
        linear_rmse,
        random_forest_rmse,
        xgboost_rmse
    ],
    "R²": [
        linear_r2,
        random_forest_r2,
        xgboost_r2
    ]
})

results

,Model,RMSE,R²
0,Linear Regression,31269.687411,0.872523
1,Random Forest,29426.843227,0.887105
2,XGBoost,28338.908942,0.895299


In [17]:
# Get the trained XGBoost model from the pipeline

xgboost_estimator = xgboost_model.named_steps["model"]

In [18]:
# Get the feature names after preprocessing

feature_names = xgboost_model.named_steps[
    "preprocessor"
].get_feature_names_out()

In [19]:
# Get XGBoost feature importance

feature_importance = pd.Series(
    xgboost_estimator.feature_importances_,
    index=feature_names
)

feature_importance = feature_importance.sort_values(
    ascending=False
)

feature_importance.head(10)

num__OverallQual          0.356853
cat__LandContour_Bnk      0.067107
num__GarageCars           0.065506
cat__GarageFinish_Unf     0.050274
cat__GarageType_Detchd    0.048231
num__GrLivArea            0.043997
cat__BsmtQual_Ex          0.027757
cat__ExterQual_Ex         0.023311
num__KitchenAbvGr         0.019306
cat__CentralAir_N         0.018932
dtype: float32

In [20]:
# Top 10 most important features

top_features = feature_importance.head(10).copy()
top_features.index = top_features.index.str.replace(
    "num__",
    "",
    regex=False
).str.replace(
    "cat__",
    "",
    regex=False
)

print("Top 10 Most Important Features:")
print(top_features)

Top 10 Most Important Features:
OverallQual          0.356853
LandContour_Bnk      0.067107
GarageCars           0.065506
GarageFinish_Unf     0.050274
GarageType_Detchd    0.048231
GrLivArea            0.043997
BsmtQual_Ex          0.027757
ExterQual_Ex         0.023311
KitchenAbvGr         0.019306
CentralAir_N         0.018932
dtype: float32
